In [ ]:
import jax
import jax.numpy as jnp
import diffrax
import matplotlib.pyplot as plt


def length(v):
    return jnp.sqrt(jnp.sum(v**2))


def normalize(v):
    L = length(v)
    return jax.lax.cond(L == 0, lambda _: jnp.zeros_like(v), lambda _: v / L, None)


def joint_vector_field(t, state, args):
    k_base, _ = args
    # Activate coupling only when 30 < t < 100
    k = k_base * jax.lax.select((t > 30.0) & (t < 100.0), 1.0, 0.0)

    s1, s2 = state
    x1, y1, z1 = s1["x"], s1["y"], s1["z"]
    x2, y2, z2 = s2["x"], s2["y"], s2["z"]

    # Master system (unperturbed conservative dynamics)
    ds1 = {
        "x": (z1 @ x1) * y1 - (y1 @ x1) * z1 - 0.4 * k * (x2 - x1),
        "y": (x1 @ y1) * z1 - (z1 @ y1) * x1,
        "z": (y1 @ z1) * x1 - (x1 @ z1) * y1,
    }

    # Slave system with conditional true ODE-based coupling term (-k * (x2 - x1)) on x component
    ds2 = {
        "x": (z2 @ x2) * y2 - (y2 @ x2) * z2 - k * (x2 - x1),
        "y": (x2 @ y2) * z2 - (z2 @ y2) * x2,
        "z": (y2 @ z2) * x2 - (x2 @ z2) * y2,
    }

    return (ds1, ds2)


def run_simulation():
    key = jax.random.PRNGKey(42)
    keys = jax.random.split(key, 6)

    s1_init = {
        "x": jax.random.normal(keys[0], (3,)),
        "y": jax.random.normal(keys[1], (3,)),
        "z": jax.random.normal(keys[2], (3,)),
    }
    s2_init = {
        "x": jax.random.normal(keys[3], (3,)),
        "y": jax.random.normal(keys[4], (3,)),
        "z": jax.random.normal(keys[5], (3,)),
    }

    term = diffrax.ODETerm(joint_vector_field)
    solver = diffrax.Tsit5()  # Robust adaptive explicit Runge-Kutta solver
    stepsize_controller = diffrax.PIDController(rtol=1e-5, atol=1e-5)

    t0, t1 = 0.0, 200.0
    ts = jnp.linspace(t0, t1, 2000)
    coupling_strength = 20.0  # High coupling strength

    y0 = (s1_init, s2_init)

    solution = diffrax.diffeqsolve(
        term,
        solver,
        t0=t0,
        t1=t1,
        dt0=0.01,
        y0=y0,
        args=(coupling_strength, None),
        stepsize_controller=stepsize_controller,
        saveat=diffrax.SaveAt(ts=ts),
        max_steps=500_000,
    )

    return ts, solution.ys

In [ ]:
times, ys = run_simulation()
s1_hist, s2_hist = ys

In [ ]:
# Compute mean squared error between all component pairs of master and slave systems over time
components = ["x", "y", "z"]
plt.figure(figsize=(12, 7))


for c1 in components:
    for c2 in components:
        error = jnp.mean((s1_hist[c1] - s2_hist[c2]) ** 2, axis=1)
        plt.title(str((c1, c2)))
        plt.plot(times, error, label=f"MSE({c1}₁, {c2}₂)", lw=1.5)
        plt.axvspan(
            30.0, 100.0, color="yellow", alpha=0.3, label="Sync Active (30 < t < 100)"
        )
        plt.show()

#    plt.yscale('log')
#    plt.title('True ODE Synchronization Error Across All Component Pairs')
#    plt.xlabel('Time')
#    plt.ylabel('Squared Error (Log Scale)')
#    plt.grid(True, which="both", ls="--")
#    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
#    plt.tight_layout()
#    plt.show()